In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-26")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-53d3ceec-2590-4510-b40a-67a9fd1adcb6;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 133ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
from pyspark.sql.types import *
messy_data = [
    ("O0001", "C001", "Electronics", 2,    1299.99, "2023-01-05"),
    ("O0002", "C002", "Electronics", 1,    None,    "2023-01-07"),
    ("O0003", None,   "Furniture",   4,    349.99,  "2023-01-10"),
    ("O0004", "C004", None,          -1,   89.99,   "2023-01-12"),
    ("O0005", "C005", "Electronics", None, 29.99,   None),
    (None,    "C006", "Furniture",   2,    -50.0,   "2023-01-18"),
    ("O0007", "C007", "Electronics", 1,    None,    "2023-01-20"),
    ("O0008", "C008", "Furniture",   0,    59.99,   "2023-01-22"),
    ("O0009", "C009", "Electronics", 1,    1299.99, "2023-01-25"),
    ("O0010", "C010", None,          2,    79.99,   "2023-01-28")
]

messy_schema = StructType([
    StructField("order_id",    StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("category",    StringType(), True),
    StructField("quantity",    IntegerType(), True),
    StructField("unit_price",  DoubleType(), True),
    StructField("order_date",  StringType(), True)
])

messy_df = spark.createDataFrame(messy_data, messy_schema) \
    .withColumn("order_date", F.to_date("order_date", "yyyy-MM-dd"))

messy_df.show()

+--------+-----------+-----------+--------+----------+----------+
|order_id|customer_id|   category|quantity|unit_price|order_date|
+--------+-----------+-----------+--------+----------+----------+
|   O0001|       C001|Electronics|       2|   1299.99|2023-01-05|
|   O0002|       C002|Electronics|       1|      NULL|2023-01-07|
|   O0003|       NULL|  Furniture|       4|    349.99|2023-01-10|
|   O0004|       C004|       NULL|      -1|     89.99|2023-01-12|
|   O0005|       C005|Electronics|    NULL|     29.99|      NULL|
|    NULL|       C006|  Furniture|       2|     -50.0|2023-01-18|
|   O0007|       C007|Electronics|       1|      NULL|2023-01-20|
|   O0008|       C008|  Furniture|       0|     59.99|2023-01-22|
|   O0009|       C009|Electronics|       1|   1299.99|2023-01-25|
|   O0010|       C010|       NULL|       2|     79.99|2023-01-28|
+--------+-----------+-----------+--------+----------+----------+



**Problem 1** | Easy

Data quality audit

Given a messy orders dataset (provided in the notebook), produce a one-row summary showing the null count for every column.

In [3]:
messy_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in messy_df.columns
    ]
).show()

+--------+-----------+--------+--------+----------+----------+
|order_id|customer_id|category|quantity|unit_price|order_date|
+--------+-----------+--------+--------+----------+----------+
|       1|          1|       2|       1|         2|         1|
+--------+-----------+--------+--------+----------+----------+



**Problem 2** | Easy

Drop rows missing critical fields only

From the messy dataset, drop rows where order_id OR customer_id is null (these are non-negotiable for the pipeline). Keep rows even if other fields are null.

In [4]:
messy_df.na.drop(
    subset=['order_id', 'customer_id']
).show()

+--------+-----------+-----------+--------+----------+----------+
|order_id|customer_id|   category|quantity|unit_price|order_date|
+--------+-----------+-----------+--------+----------+----------+
|   O0001|       C001|Electronics|       2|   1299.99|2023-01-05|
|   O0002|       C002|Electronics|       1|      NULL|2023-01-07|
|   O0004|       C004|       NULL|      -1|     89.99|2023-01-12|
|   O0005|       C005|Electronics|    NULL|     29.99|      NULL|
|   O0007|       C007|Electronics|       1|      NULL|2023-01-20|
|   O0008|       C008|  Furniture|       0|     59.99|2023-01-22|
|   O0009|       C009|Electronics|       1|   1299.99|2023-01-25|
|   O0010|       C010|       NULL|       2|     79.99|2023-01-28|
+--------+-----------+-----------+--------+----------+----------+



**Problem 3** | Medium

Smart fill — use category average for missing prices

For rows where unit_price is null, fill it with the average unit_price for that product's category (not a single global constant). This requires computing per-group averages and joining them back.

In [5]:
category_avg = messy_df.groupBy("category").agg(
    F.round(F.avg("unit_price"), 2).alias("category_avg_price")
)

filled = messy_df.join(category_avg, on="category", how="left")\
    .withColumn(
        "unit_price_filled",
        F.coalesce(F.col("unit_price"), F.col("category_avg_price"))
    )

filled.select("order_id", "category", "unit_price", "category_avg_price", "unit_price_filled").show()

+--------+-----------+----------+------------------+-----------------+
|order_id|   category|unit_price|category_avg_price|unit_price_filled|
+--------+-----------+----------+------------------+-----------------+
|   O0001|Electronics|   1299.99|            876.66|          1299.99|
|   O0002|Electronics|      NULL|            876.66|           876.66|
|   O0003|  Furniture|    349.99|            119.99|           349.99|
|   O0004|       NULL|     89.99|              NULL|            89.99|
|   O0005|Electronics|     29.99|            876.66|            29.99|
|    NULL|  Furniture|     -50.0|            119.99|            -50.0|
|   O0007|Electronics|      NULL|            876.66|           876.66|
|   O0008|  Furniture|     59.99|            119.99|            59.99|
|   O0010|       NULL|     79.99|              NULL|            79.99|
|   O0009|Electronics|   1299.99|            876.66|          1299.99|
+--------+-----------+----------+------------------+-----------------+



**Problem 4** | Medium

Flag suspicious records

Flag any order as "suspicious" if quantity is negative or zero, OR unit_price is negative, OR order_date is null. Otherwise flag as "valid". Count how many fall into each bucket.

In [ ]:
flagged_df=messy_df.withColumn('Flag',
F.when((F.col('quantity')<=0)
       |(F.col('unit_price')<=0)|
       (F.col('order_date').isNull()),'suspicious').otherwise('valid')
)
flagged_df.show(4,truncate=False)
flagged_df.groupBy('Flag').agg(
    F.count('*').alias('count')
).show()


+--------+-----------+-----------+--------+----------+----------+----------+
|order_id|customer_id|category   |quantity|unit_price|order_date|Flag      |
+--------+-----------+-----------+--------+----------+----------+----------+
|O0001   |C001       |Electronics|2       |1299.99   |2023-01-05|valid     |
|O0002   |C002       |Electronics|1       |NULL      |2023-01-07|valid     |
|O0003   |NULL       |Furniture  |4       |349.99    |2023-01-10|valid     |
|O0004   |C004       |NULL       |-1      |89.99     |2023-01-12|suspicious|
+--------+-----------+-----------+--------+----------+----------+----------+
only showing top 4 rows
+----------+-----+
|      Flag|count|
+----------+-----+
|     valid|    6|
|suspicious|    4|
+----------+-----+



In [10]:
crm_data = [
    ("C001", "James Anderson", None,              "Enterprise"),
    ("C002", None,              "310-555-0102",   None),
    ("C003", "Robert Johnson",  "312-555-0103",   "Enterprise")
]
web_data = [
    ("C001", "James A.",        "212-555-0101",   "Enterprise"),
    ("C002", "Maria Garcia",    None,              "SMB"),
    ("C003", "Robert Johnson",  None,              None)
]

crm_df = spark.createDataFrame(crm_data, ["customer_id", "name", "phone", "segment"])
web_df = spark.createDataFrame(web_data, ["customer_id", "name", "phone", "segment"])

print("CRM source:")
crm_df.show()
print("Web source:")
web_df.show()

CRM source:
+-----------+--------------+------------+----------+
|customer_id|          name|       phone|   segment|
+-----------+--------------+------------+----------+
|       C001|James Anderson|        NULL|Enterprise|
|       C002|          NULL|310-555-0102|      NULL|
|       C003|Robert Johnson|312-555-0103|Enterprise|
+-----------+--------------+------------+----------+

Web source:
+-----------+--------------+------------+----------+
|customer_id|          name|       phone|   segment|
+-----------+--------------+------------+----------+
|       C001|      James A.|212-555-0101|Enterprise|
|       C002|  Maria Garcia|        NULL|       SMB|
|       C003|Robert Johnson|        NULL|      NULL|
+-----------+--------------+------------+----------+



**Problem 5** | Hard

Reconcile two data sources

You have two versions of the customers table — one from the CRM, one from the web signup form — each with some fields populated and others null. Merge them into a single clean record per customer, preferring the CRM source but falling back to the web source field-by-field using coalesce().

In [15]:
crm = crm_df.alias("crm")
web = web_df.alias("web")

merged = (crm.join(web, on="customer_id", how="outer")
    .select(
        F.col("customer_id"),
        F.coalesce(F.col("crm.name"),    F.col("web.name")).alias("name"),
        F.coalesce(F.col("crm.phone"),   F.col("web.phone")).alias("phone"),
        F.coalesce(F.col("crm.segment"), F.col("web.segment")).alias("segment")
    ))

merged.show()

+-----------+--------------+------------+----------+
|customer_id|          name|       phone|   segment|
+-----------+--------------+------------+----------+
|       C001|James Anderson|212-555-0101|Enterprise|
|       C002|  Maria Garcia|310-555-0102|       SMB|
|       C003|Robert Johnson|312-555-0103|Enterprise|
+-----------+--------------+------------+----------+



**Problem 6** | Hard

Build a data quality scorecard

For the messy orders dataset, calculate a completeness percentage per column: (non-null count / total count) * 100, rounded to 1 decimal. Return one row per column with column_name and completeness_pct, sorted ascending (worst first).

In [16]:
total = messy_df.count()

scorecard_rows = []
for c in messy_df.columns:
    non_null_count = messy_df.filter(F.col(c).isNotNull()).count()
    completeness = round(100.0 * non_null_count / total, 1)
    scorecard_rows.append((c, completeness))

scorecard_df = spark.createDataFrame(scorecard_rows, ["column_name", "completeness_pct"])
scorecard_df.orderBy("completeness_pct").show()

+-----------+----------------+
|column_name|completeness_pct|
+-----------+----------------+
| unit_price|            80.0|
|   category|            80.0|
|   quantity|            90.0|
| order_date|            90.0|
|customer_id|            90.0|
|   order_id|            90.0|
+-----------+----------------+

